# ProteinMPNN Inverse Folding

![ProteinMPNN Inverse Folding](https://proto-bio.github.io/proto-assets/images/tool/proteinmpnn/hero.png)

This notebook demonstrates the four ProteinMPNN tools. First, we use `run_proteinmpnn_sample` to generate new amino acid sequences conditioned on a target backbone structure. Second, we use `run_proteinmpnn_score` to evaluate how well an existing sequence fits a given structure by computing structure-conditioned likelihoods. Third, we use `run_proteinmpnn_sample_homooligomer` to design a symmetric homooligomer by tying corresponding positions across its protomers. Fourth, we use `run_proteinmpnn_gradient` to differentiate ProteinMPNN's mean-NLL objective with respect to a continuous `(L, 20)` distribution — useful as a structure-conditioned loss inside MCMC or gradient descent over relaxed sequences.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("proteinmpnn")
display_overview("proteinmpnn")
display_docs_section("proteinmpnn", "Background")

## Available tools

In [ ]:
display_available_tools("proteinmpnn")

### `run_proteinmpnn_sample`

ProteinMPNN designs new protein sequences that are predicted to fold into a target backbone structure. We use Green Fluorescent Protein (GFP) as our target, a 238-residue beta-barrel protein whose well-characterized fold makes it an ideal test case for inverse folding. ProteinMPNN analyzes the 3D backbone coordinates and autoregressively generates amino acid sequences optimized for the target shape. The model also supports antibody-optimized weights via the `abmpnn` model choice.

In [ ]:
from proto_tools import (
    InverseFoldingStructureInput,
    run_proteinmpnn_sample,
    ProteinMPNNSampleInput,
    ProteinMPNNSampleConfig,
    Complex,
)
from proto_tools.entities.structures.examples import get_gfp_structure

In [ ]:
# Display input docs
display_api_reference("proteinmpnn", "input", "run_proteinmpnn_sample")

# Load the GFP structure and wrap it in an inverse folding input
gfp_structure = get_gfp_structure()
struct_input = InverseFoldingStructureInput(structure=gfp_structure, chains_to_redesign=["A"])
inputs = ProteinMPNNSampleInput(inputs=[struct_input])

In [ ]:
# Display config docs
display_api_reference("proteinmpnn", "config", "run_proteinmpnn_sample")

# Configure sampling | see docs above for all fields
config = ProteinMPNNSampleConfig(
    num_sequences_per_structure=2,  # Generate 2 sequence designs
    temperature=0.1,                # Conservative designs (closer to consensus)
    device="cuda",                  # Change to "cpu" if no GPU available
)

In [ ]:
# Run the sampling function
result = run_proteinmpnn_sample(inputs, config)

In [ ]:
# Display output docs
display_api_reference("proteinmpnn", "output", "run_proteinmpnn_sample")

# Walk the structured output: one ProteinMPNNDesignSet per input structure,
# each holding one ProteinMPNNDesign (a complete complex) per requested sequence.
for i, design_set in enumerate(result.design_sets):
    print(f"Structure {i} complexes:")
    for j, design in enumerate(design_set.complexes, 1):
        print(f"  Design {j}:")
        # chain_sequence_map() covers every chain of the input (designed + fixed context)
        for chain_id, seq in design.chain_sequence_map().items():
            preview = f"{seq[:50]}..." if len(seq) > 50 else seq
            designed = any(c.id == chain_id and d for c, d in zip(design.chains, design.designed, strict=True))
            tag = "designed" if designed else "context"
            print(f"    Chain {chain_id} ({tag}, {len(seq)} aa): {preview}")
        print(f"    perplexity={design.metrics['perplexity']:.3f}  "
              f"sequence_recovery={design.metrics['sequence_recovery']:.3f}")

### Design then fold

A `ProteinMPNNDesign` is a self-contained complex, so `design` accepts it directly. This lets you hand a design straight to a structure predictor (for example to validate that the designed sequence folds back to the target backbone) without manually reconstructing chains.

In [ ]:
# A design can be passed straight into a structure predictor input
best_design = result.design_sets[0].complexes[0]
fold_complex = best_design
print(f"Complex chains: {fold_complex.num_chains()}")
print(f"Chain sequences: {[s[:30] + '...' for s in fold_complex.chain_sequences]}")
# fold_complex is now ready to pass to ESMFold / AlphaFold / Boltz, etc.

### `run_proteinmpnn_sample_homooligomer`

`run_proteinmpnn_sample` designs each chain independently, so the copies of a homooligomer come
out with different sequences and each protomer was chosen against placeholder partners rather
than against the interface it actually packs into. `run_proteinmpnn_sample_homooligomer` ties
corresponding positions across the protomers instead: one protomer sequence is sampled while the
whole oligomer is in context, then returned on every protomer.

Its selections are **per protomer**, not per chain. `protomer_chains` lists the chains to tie (at
least two, all the same length) and `protomer_fixed_positions` is a flat list of positions
*within one protomer*, applied to all of them. There is deliberately no per-chain
`fixed_positions`: tied decoding averages the per-position biases across the tied group, so a
position fixed on only some protomers would silently fix it on all of them.

Note that `temperature` is not comparable between the two tools. Tying averages the per-protomer
logits together with their sampling noise, so an n-mer is less diverse than a monomer at the same
nominal value.

1TNF is the TNF-alpha homotrimer, the same structure Caliby's notebook uses for its equivalent
`symmetric_chains` feature. The cell below designs it both ways to show the difference.

In [ ]:
from proto_tools import (
    ProteinMPNNHomooligomerSampleConfig,
    ProteinMPNNHomooligomerSampleInput,
    ProteinMPNNHomooligomerStructureInput,
    run_proteinmpnn_sample_homooligomer,
)
from proto_tools.entities.structures import Structure

In [ ]:
# Display input docs
display_api_reference("proteinmpnn", "input", "run_proteinmpnn_sample_homooligomer")

# 1TNF assembly 1: three identical chains.
trimer = Structure(structure="https://files.rcsb.org/download/1TNF-assembly1.cif")
print(f"Chains {trimer.get_chain_ids()}, identical: {len(set(trimer.get_chain_sequences().values())) == 1}")

homo_inputs = ProteinMPNNHomooligomerSampleInput(
    inputs=[ProteinMPNNHomooligomerStructureInput(structure=trimer, protomer_chains=["A", "B", "C"])]
)

In [ ]:
# Display config docs
display_api_reference("proteinmpnn", "config", "run_proteinmpnn_sample_homooligomer")

# Configure tied sampling | see docs above for all fields
homo_config = ProteinMPNNHomooligomerSampleConfig(
    num_sequences_per_structure=1,
    temperature=0.1,
    seed=42,
    device="cuda",  # Change to "cpu" if no GPU available
)

In [ ]:
# Design the same trimer both ways to compare.
# The untied run logs a warning pointing at this tool, because its chains are sequence-identical.
untied = run_proteinmpnn_sample(
    ProteinMPNNSampleInput(inputs=[InverseFoldingStructureInput(structure=trimer)]),
    ProteinMPNNSampleConfig(num_sequences_per_structure=1, temperature=0.1, seed=42, device="cuda"),
)
tied = run_proteinmpnn_sample_homooligomer(homo_inputs, homo_config)

for label, run_output in (("independent", untied), ("tied", tied)):
    protomer_design = run_output.design_sets[0].complexes[0]
    sequences = list(protomer_design.chain_sequence_map().values())
    print(f"{label:12s} identical protomers: {len(set(sequences)) == 1}")
    for chain_id, seq in protomer_design.chain_sequence_map().items():
        print(f"    Chain {chain_id}: {seq[:50]}{'...' if len(seq) > 50 else ''}")

In [ ]:
# Display output docs | the output type is shared with run_proteinmpnn_sample
display_api_reference("proteinmpnn", "output", "run_proteinmpnn_sample_homooligomer")

tied_design = tied.design_sets[0].complexes[0]
print(f"perplexity={tied_design.metrics['perplexity']:.3f}  "
      f"sequence_recovery={tied_design.metrics['sequence_recovery']:.3f}")

# Fixing a position is per protomer, and applies to every protomer chain.
fixed_inputs = ProteinMPNNHomooligomerSampleInput(
    inputs=[
        ProteinMPNNHomooligomerStructureInput(
            structure=trimer,
            protomer_chains=["A", "B", "C"],
            protomer_fixed_positions=[10, 11],  # 1-indexed within one protomer
        )
    ]
)
print(f"Broadcast to every protomer: {fixed_inputs.inputs[0].fixed_positions.chains}")

### `run_proteinmpnn_score`

The scoring function evaluates how well a protein sequence fits a given backbone structure. It computes per-residue log-likelihoods using ProteinMPNN's structure-conditioned language model, producing metrics such as perplexity and average log-likelihood. Lower perplexity indicates better sequence-structure compatibility, making this useful for ranking candidate sequences or evaluating natural sequences against their native structures.

Here we score one of the designed sequences from the sampling step above to evaluate how well it fits the GFP backbone.

In [ ]:
from proto_tools import (
    ProteinMPNNScoringInput,
    ProteinMPNNScoringConfig,
    SequenceStructurePair,
    run_proteinmpnn_score,
)

In [ ]:
# Display input docs
display_api_reference("proteinmpnn", "input", "run_proteinmpnn_score")

# Take the first design's redesigned chain A sequence and score it
first_design = result.design_sets[0].complexes[0]
designed_seq = first_design.designed_chains[0].sequence
pair = SequenceStructurePair(sequence=designed_seq, structure=gfp_structure)
scoring_input = ProteinMPNNScoringInput(sequence_structure_pairs=[pair])

In [ ]:
# Display config docs
display_api_reference("proteinmpnn", "config", "run_proteinmpnn_score")

# Configure scoring | see docs above for all fields
scoring_config = ProteinMPNNScoringConfig(
    device="cuda",  # Change to "cpu" if no GPU available
)

In [ ]:
# Run the scoring function
score_result = run_proteinmpnn_score(scoring_input, scoring_config)

In [ ]:
# Display output docs
display_api_reference("proteinmpnn", "output", "run_proteinmpnn_score")

# Display scoring metrics for the designed sequence
score = score_result.scores[0]
print(f"Perplexity: {score['perplexity']:.3f}")
print(f"Log-likelihood: {score['log_likelihood']:.3f}")
print(f"Avg log-likelihood: {score['avg_log_likelihood']:.3f}")

### `run_proteinmpnn_gradient`

The gradient tool takes a relaxed `(L, 20)` distribution over amino acids together with a backbone structure and returns the gradient of ProteinMPNN's structure-conditioned mean-NLL objective with respect to those input logits. Use this as a differentiable, structure-conditioned loss inside MCMC or gradient descent over relaxed protein sequences. Set `compute_gradient=False` to get the same scalar objective without the backward pass — useful for scoring proposals.

In [ ]:
from proto_tools import (
    ProteinMPNNGradientConfig,
    ProteinMPNNGradientInput,
    run_proteinmpnn_gradient,
)
from proto_tools.utils import one_hot_protein_logits

In [ ]:
# Display input docs
display_api_reference("proteinmpnn", "input", "run_proteinmpnn_gradient")

# A gradient is reported per position, so this tool rejects structures holding residues its
# parser cannot read (GFP's CRO chromophore is one). Seed from the all-canonical in-tree
# fixture instead; sharpness=2.0 yields a biased-but-not-saturated softmax target.
from pathlib import Path

_repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
gradient_structure = Structure.from_file(
    str(_repo_root / "proto_tools" / "tools" / "inverse_folding" / "example_input_fixture.pdb")
)
logits = one_hot_protein_logits(gradient_structure.get_chain_sequence("A"), sharpness=2.0)
# Each candidate bundles its relaxed sequence with the backbone it is scored against
gradient_input = ProteinMPNNGradientInput(
    inputs={
        "logits": logits,
        "structure": gradient_structure,
        "chains_to_redesign": ["A"],
    },
)

In [ ]:
# Display config docs
display_api_reference("proteinmpnn", "config", "run_proteinmpnn_gradient")

# Configure the gradient tool | see docs above for all fields
gradient_config = ProteinMPNNGradientConfig(
    model_choice="proteinmpnn",
    hard=0.0,                # Pure softmax; set hard=1.0 for a straight-through estimator
    compute_gradient=True,   # Set False for forward-only NLL scoring
    device="cuda",           # Change to "cpu" if no GPU available
)

In [ ]:
# Run the gradient function
gradient_result = run_proteinmpnn_gradient(gradient_input, gradient_config)

# Display output docs
display_api_reference("proteinmpnn", "output", "run_proteinmpnn_gradient")

# One result per candidate; this call passed one, so results holds one
candidate = gradient_result.results[0]

print(f"Mean NLL:           {candidate.loss:.3f}")
print(f"Perplexity:         {candidate.metrics['perplexity']:.3f}")
print(f"Avg log-likelihood: {candidate.metrics['avg_log_likelihood']:.3f}")
assert candidate.gradient is not None  # backward pass enabled
print(f"Gradient shape:     ({len(candidate.gradient)}, {len(candidate.gradient[0])})")
print(f"Gradient row 0 (first 5 cols): {[round(v, 4) for v in candidate.gradient[0][:5]]}")

## Export Results

Designed sequences can be saved to standard file formats for downstream analysis, gene synthesis, or further computational evaluation such as structure prediction or molecular dynamics.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="proteinmpnn_designs", export_path=output_dir, file_format="fasta")
print(f"Sampling results exported to {output_dir / 'proteinmpnn_designs.fasta'}")

score_result.export(name="proteinmpnn_scores", export_path=output_dir, file_format="csv")
print(f"Scoring results exported to {output_dir / 'proteinmpnn_scores.csv'}")